# 07 - Types of images, colour models, pseudo- and full-colour processing

**Syllabus:** types of images: binary, grayscale, colour · colour representation: colour models ·
pseudo-colour and full-colour image processing

Colour is where "an image is just an array" starts to need qualification: the *same* array means
different things depending on which model you declare it to be in, and choosing the wrong model
makes easy tasks hard. Skin detection is a nightmare in RGB and nearly trivial in HSV.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
# the graders are the test harness -- everything else is written out in full
from cvlab import check_pseudocolor, check_rgb_to_hsi, check_rgb_to_ycbcr

rgb = cv2.cvtColor(cv2.imread("images/fruits.jpg"), cv2.COLOR_BGR2RGB)

## 1. Types of images

| type | shape | dtype | values | bytes/pixel |
|---|---|---|---|---|
| binary | (H, W) | uint8 or bool | {0, 255} / {0,1} | 1 (1/8 if bit-packed) |
| grayscale | (H, W) | uint8 | 0..255 | 1 |
| colour | (H, W, 3) | uint8 | 0..255 per channel | 3 |
| indexed / palette | (H, W) + LUT | uint8 | index into a 256×3 table | 1 + table |
| multispectral | (H, W, K) | uint16 often | sensor-dependent | 2K |

An **indexed** image is the bridge to pseudo-colour: the pixel value is not an intensity at all,
it is a *row number* in a colour table.

In [ ]:
import cv2
binr = ((cv2.imread("images/sudoku.png", cv2.IMREAD_GRAYSCALE) > 128) * 255).astype(np.uint8)
gray = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)
for name, im in [("binary", binr), ("grayscale", gray), ("colour", rgb)]:
    print(f"{name:10} shape={str(im.shape):18} dtype={im.dtype}  "
          f"distinct values={len(np.unique(im)):5}  bytes={im.nbytes:,}")

## 2. Colour models

A colour model is a coordinate system for colour. Each one exists because some task is easy in it.

| model | axes | good for | note |
|---|---|---|---|
| **RGB** | red, green, blue | displays, sensors | additive; device-dependent |
| **CMY / CMYK** | cyan, magenta, yellow (+ black) | printing | subtractive; `C = 1 - R` |
| **HSI / HSV / HSL** | hue, saturation, intensity/value/lightness | *describing* colour, thresholding by colour | decouples colour from brightness |
| **YCbCr** | luma + 2 chroma | JPEG, MPEG, video | lets you subsample chroma cheaply |
| **CIE L\*a\*b\*** | lightness + 2 opponent axes | perceptual distance, colour difference | roughly uniform to the eye |
| **CIE XYZ** | device-independent primaries | the reference all others convert through | not directly displayable |

The key structural difference: RGB mixes brightness into all three channels, so changing the
lighting changes all three. HSI, YCbCr and Lab all separate a brightness axis from colour axes —
which is why they dominate anywhere lighting varies.

In [ ]:
# RGB is additive; CMY is its my_bitwise_not. One line each direction.
cmy = 255 - rgb
_imgs, _titles = [rgb, cmy], ["RGB", "CMY = 255 - RGB"]
fig, axes = plt.subplots(1, 2, figsize=(4 * 2, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

# Same image, four models, channel by channel.
bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
models = {
    "RGB":  (rgb, ["R", "G", "B"]),
    "HSV":  (cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV), ["H", "S", "V"]),
    "YCrCb":(cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb), ["Y", "Cr", "Cb"]),
    "Lab":  (cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB), ["L", "a", "b"]),
}
fig, axes = plt.subplots(4, 4, figsize=(13, 13))
for row, (name, (im, chans)) in enumerate(models.items()):
    axes[row,0].imshow(rgb); axes[row,0].set_title(f"{name}: original", fontsize=9)
    for k in range(3):
        axes[row,k+1].imshow(im[..., k], cmap="gray", vmin=0, vmax=255)
        axes[row,k+1].set_title(f"{name} - {chans[k]}", fontsize=9)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

Look at the **V**, **Y** and **L** channels: all three are recognisable photographs. Now look at
**S**, **Cr/Cb** and **a/b**: those are nearly featureless. That observation is worth money —
it is the entire justification for chroma subsampling in JPEG, and it says most of an image's
*information* lives in its brightness.

### Task 1 - RGB to HSI

The Gonzalez HSI model. With R, G, B normalised to [0,1]:

$$I = \frac{R+G+B}{3} \qquad S = 1 - \frac{\min(R,G,B)}{I} \qquad
H = \begin{cases}\theta & B \le G\\ 360 - \theta & B > G\end{cases}$$

$$\theta = \arccos\left[\frac{\tfrac{1}{2}\left[(R-G) + (R-B)\right]}
{\sqrt{(R-G)^2 + (R-B)(G-B)}}\right]$$

Return `H` in degrees `[0, 360)`, `S` and `I` in `[0, 1]`, as three float arrays.

**Two guards you must write**, and the grader tests both: for a black pixel `I = 0` so the `S`
denominator vanishes, and for any grey pixel the `θ` denominator vanishes. Neither may produce
`NaN`. Hue is undefined when `S = 0` — return 0 there.

In [ ]:
def my_cvtColor_HSI(rgb):
    """Convert RGB to the Gonzalez HSI model.

    Parameters
    ----------
    rgb : (H, W, 3) uint8 array

    Returns
    -------
    (H, S, I) tuple of three (H, W) float arrays.
    H in degrees [0, 360), S in [0, 1], I in [0, 1]. No NaNs, ever:
    set H = 0 wherever S == 0, and S = 0 wherever I == 0.
    """
    # TODO: normalise to 0..1, then I, then S, then theta, then H
    raise NotImplementedError

In [ ]:
check_rgb_to_hsi(my_cvtColor_HSI)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV) -- but HSV is NOT HSI. Know the difference:
#   HSI: I = (R+G+B)/3          (what you just implemented, Gonzalez)
#   HSV: V = max(R,G,B)         (what OpenCV gives you)
#   HSL: L = (max+min)/2
# And OpenCV squeezes H into 0..179 for uint8 (so it fits a byte) -- NOT 0..359.
import cv2, numpy as np
bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
H, S, I = my_cvtColor_HSI(rgb)
print("cv2 H range:", hsv[...,0].min(), "..", hsv[...,0].max(), " <- degrees/2 !")
print("your H range:", round(H.min(), 1), "..", round(H.max(), 1))
print("\nafter doubling cv2's H, mean absolute hue difference:",
      round(np.abs(hsv[...,0].astype(float)*2 - H).mean(), 2), "degrees")
print("cv2 V (max) vs your I (mean) at a pure-red pixel: V=255, I=%.0f" % (I.max()*255))
print("\nUse cv2.inRange(hsv, lower, upper) to threshold in HSV -- the standard colour-mask idiom.")
mask = cv2.inRange(hsv, np.array([0,90,60]), np.array([10,255,255])) | \
       cv2.inRange(hsv, np.array([170,90,60]), np.array([179,255,255]))
_imgs, _titles = [rgb, mask], ["input", "cv2.inRange red mask (hue wraps, so TWO ranges)"]
fig, axes = plt.subplots(1, 2, figsize=(4 * 2, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Work in `float64` from the start: `f = rgb.astype(np.float64) / 255.0`, then unpack
`R, G, B = f[...,0], f[...,1], f[...,2]`.

For the two zero-denominator guards, `np.where` alone is not enough — both branches get
evaluated, so the division still runs and still warns/produces NaN. Use
`np.divide(num, den, out=np.zeros_like(num), where=den > 1e-12)`, which only computes
where the condition holds and leaves zeros elsewhere.

Clip the arccos argument to [-1, 1] before calling it — floating point will hand you
1.0000000000000002 and `arccos` returns NaN for that.

</details>

In [ ]:
H, S, I = my_cvtColor_HSI(rgb)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
axes[0].imshow(rgb); axes[0].set_title("RGB")
im = axes[1].imshow(H, cmap="hsv", vmin=0, vmax=360); axes[1].set_title("H (degrees)")
plt.colorbar(im, ax=axes[1], fraction=.046)
axes[2].imshow(S, cmap="gray", vmin=0, vmax=1); axes[2].set_title("S")
axes[3].imshow(I, cmap="gray", vmin=0, vmax=1); axes[3].set_title("I")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

# The payoff: pick out the red fruit by HUE, which RGB thresholds cannot do cleanly.
mask = (((H < 20) | (H > 340)) & (S > 0.35)).astype(np.uint8) * 255
naive = ((rgb[...,0] > 150) & (rgb[...,1] < 100)).astype(np.uint8) * 255
_imgs, _titles = [rgb, mask, naive], ["input", "hue-based red mask", "naive RGB threshold"]
fig, axes = plt.subplots(1, 3, figsize=(4 * 3, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

### Task 2 - RGB to YCbCr

The JPEG full-range BT.601 transform. This is a pure matrix multiply plus an offset:

$$\begin{aligned}
Y &= 0.299R + 0.587G + 0.114B\\
C_b &= 128 - 0.168736R - 0.331264G + 0.5B\\
C_r &= 128 + 0.5R - 0.418688G - 0.081312B
\end{aligned}$$

Note `Y` is exactly the luma grayscale from notebook 02. The `+128` offset exists because the
chroma terms are signed and `uint8` is not: grey means "no colour", which lands at 128, not 0.

Return `(H, W, 3)` uint8 in **Y, Cb, Cr** order. (OpenCV returns Y, **Cr**, Cb — the grader
checks against it channel by channel, so mind the swap.)

In [ ]:
def my_cvtColor_YCrCb(rgb):
    """Convert RGB to JPEG full-range BT.601 YCbCr.

    Parameters
    ----------
    rgb : (H, W, 3) uint8 array

    Returns
    -------
    (H, W, 3) uint8 array stacked as Y, Cb, Cr. Rounded and clipped to 0..255.
    A grey input pixel must give Cb = Cr = 128.
    """
    # TODO: float maths per channel, stack with np.dstack, round, clip, cast
    raise NotImplementedError

In [ ]:
check_rgb_to_ycbcr(my_cvtColor_YCrCb)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb) -- note the ORDER: Y, Cr, Cb.
# Your function returns Y, Cb, Cr. Mixing these up is a very common bug.
import cv2, numpy as np
bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
ref = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)
mine = my_cvtColor_YCrCb(rgb)
print("Y  max diff:", int(np.abs(ref[...,0].astype(int) - mine[...,0].astype(int)).max()))
print("Cb max diff:", int(np.abs(ref[...,2].astype(int) - mine[...,1].astype(int)).max()), "(cv2 index 2)")
print("Cr max diff:", int(np.abs(ref[...,1].astype(int) - mine[...,2].astype(int)).max()), "(cv2 index 1)")
print("\ncv2.split(img) -> (c0, c1, c2)   and   cv2.merge([c0, c1, c2]) -> img")
y, cr, cb = cv2.split(ref)
print("after split, shapes:", y.shape, cr.shape, cb.shape,
      "| merged back equal:", np.array_equal(cv2.merge([y, cr, cb]), ref))

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Straight transcription of the three formulas on float planes, then
`np.dstack([Y, Cb, Cr]).round().clip(0, 255).astype(np.uint8)`.

Order matters: `.round()` before `.clip()` before `.astype()`. Casting first truncates,
and skipping the clip lets a saturated colour wrap around to the opposite value.

</details>

In [ ]:
# Why video codecs subsample chroma: destroy the chroma and almost nothing happens.
ycc = my_cvtColor_YCrCb(rgb)
def rebuild(y, cb, cr):
    Y, Cb, Cr = y.astype(np.float64), cb.astype(np.float64)-128, cr.astype(np.float64)-128
    R = Y + 1.402*Cr; G = Y - 0.344136*Cb - 0.714136*Cr; B = Y + 1.772*Cb
    return np.dstack([R,G,B]).round().clip(0,255).astype(np.uint8)

def crush(ch, f):     # subsample then blow back up = what 4:2:0 does
    small = ch[::f, ::f]
    return cv2.resize(small, (ch.shape[1], ch.shape[0]), interpolation=cv2.INTER_LINEAR)

Y, Cb, Cr = ycc[...,0], ycc[...,1], ycc[...,2]
_imgs, _titles = [rgb, rebuild(Y, crush(Cb,2), crush(Cr,2)), rebuild(Y, crush(Cb,8), crush(Cr,8)), rebuild(crush(Y,8), Cb, Cr)], ["original", "chroma at 1/2 (4:2:0)", "chroma at 1/8", "LUMA at 1/8 (compare!)"]
fig, axes = plt.subplots(1, 4, figsize=(3.5 * 4, 3.5), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

Chroma at 1/8 resolution is nearly invisible. Luma at 1/8 is destroyed. Same number of bytes
thrown away, wildly different perceptual cost — that asymmetry *is* JPEG's first move, and you
will implement the rest of it in notebook 16.

---
## 3. Pseudo-colour processing

**Pseudo-colour** assigns colours to a *grayscale* image. There is no colour in the data; you add
it so the eye can distinguish more levels — humans resolve maybe 30 grey shades but thousands of
colours. Standard in thermal imaging, medical scans, elevation maps, and any depth map.

Two flavours: **intensity slicing** (bands → flat colours, what you will write) and
**colour transformation** (three smooth functions of intensity, one per channel — a colormap).

### Task 3 - intensity slicing

In [ ]:
def my_applyColorMap(gray, bounds, colors):
    """Map intensity bands to flat colours.

    Parameters
    ----------
    gray   : (H, W) uint8 array
    bounds : ascending list of ints, the band edges. May be empty.
    colors : list of (r, g, b) tuples, length len(bounds) + 1

    Returns
    -------
    (H, W, 3) uint8 RGB image. A pixel v takes colors[k] where k is the number of
    bounds strictly less than v -- i.e. band k covers bounds[k-1] < v <= bounds[k].
    Only the given colours may appear; nothing is blended.
    """
    # TODO: start from a zero RGB canvas, then one masked assignment per band
    raise NotImplementedError

In [ ]:
check_pseudocolor(my_applyColorMap)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.applyColorMap(src, colormap) -> BGR image.  Remember it returns BGR,
# so slice [..., ::-1] before handing it to matplotlib.
import cv2, numpy as np
therm = cv2.imread("images/board.jpg", cv2.IMREAD_GRAYSCALE)
print("built-in maps: COLORMAP_JET, VIRIDIS, HOT, BONE, TURBO, INFERNO, PLASMA, MAGMA, ...")
# A custom colormap is a (256,1,3) uint8 LUT -- which is exactly your banded table:
lut = np.zeros((256, 1, 3), np.uint8)
for i, (lo, hi, col) in enumerate([(0,40,(90,0,0)), (41,80,(200,90,0)), (81,120,(190,190,0)),
                                   (121,160,(0,220,240)), (161,200,(0,120,240)), (201,255,(0,0,200))]):
    lut[lo:hi+1, 0] = col                       # BGR order for cv2
custom = cv2.applyColorMap(therm, lut)
_imgs, _titles = [pseudo := my_applyColorMap(therm, [40,80,120,160,200],
        [(0,0,90),(0,90,200),(0,190,190),(240,220,0),(240,120,0),(200,0,0)]), custom[..., ::-1], cv2.applyColorMap(therm, cv2.COLORMAP_TURBO)[..., ::-1]], ["yours", "cv2.applyColorMap + custom LUT", "cv2 COLORMAP_TURBO"]
fig, axes = plt.subplots(1, 3, figsize=(3.6 * 3, 3.6), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Allocate `out = np.zeros(gray.shape + (3,), np.uint8)`, then walk the bands keeping a
running `lo`, starting at `-1` so that intensity 0 falls in band 0:

```
lo = -1
for i, hi in enumerate(list(bounds) + [255]):
    out[(gray > lo) & (gray <= hi)] = colors[i]
    lo = hi
```

Assigning a 3-tuple into a boolean-masked (H,W,3) array broadcasts across the channel axis.

</details>

In [ ]:
import cv2
therm = cv2.imread("images/board.jpg", cv2.IMREAD_GRAYSCALE)
sliced = my_applyColorMap(therm, [40, 80, 120, 160, 200],
                           [(0,0,90),(0,90,200),(0,190,190),(240,220,0),(240,120,0),(200,0,0)])
_imgs, _titles = [therm, sliced, cv2.applyColorMap(therm, cv2.COLORMAP_JET)[..., ::-1], cv2.applyColorMap(therm, cv2.COLORMAP_VIRIDIS)[..., ::-1]], ["grayscale", "your 6-band slicing", "cv2 JET (smooth)", "cv2 VIRIDIS"]
fig, axes = plt.subplots(1, 4, figsize=(3.5 * 4, 3.5), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

Your discrete slicing makes iso-intensity contours pop out — good for reading off *levels*.
The smooth colormaps preserve ordering better — good for reading *shape*. And JET, despite being
everywhere, is a bad default: it is not perceptually uniform, so it invents visual edges where the
data is smooth. VIRIDIS was designed to fix exactly that.

---
## 4. Full-colour processing

Now the input itself is colour. Two strategies, and knowing when they differ is the point:

- **per-channel** — run a grayscale operation on R, G, B independently. Simple, and it
  **shifts hue** whenever the operation is non-linear, because the three channels move by
  different amounts.
- **vector** — treat each pixel as a 3-vector and operate on it as a unit. Or: convert to a model
  with a brightness axis, process *only* that axis, convert back. Colour is preserved by construction.

Watch a per-channel histogram equalisation wreck the colours:

In [ ]:
# per-channel equalisation vs equalising the intensity axis only
per_ch = np.dstack([cv2.equalizeHist(rgb[..., k]) for k in range(3)])

hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
hsv2 = hsv.copy(); hsv2[..., 2] = cv2.equalizeHist(hsv[..., 2])
v_only = cv2.cvtColor(cv2.cvtColor(hsv2, cv2.COLOR_HSV2BGR), cv2.COLOR_BGR2RGB)

_imgs, _titles = [rgb, per_ch, v_only], ["original", "per-channel EQ (hue shifts)", "V-channel EQ only"]
fig, axes = plt.subplots(1, 3, figsize=(4 * 3, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

Hh, _, _ = my_cvtColor_HSI(rgb)
Hp, _, _ = my_cvtColor_HSI(per_ch)
Hv, _, _ = my_cvtColor_HSI(v_only)
print(f"mean |hue change|, per-channel: {np.abs(Hp-Hh).mean():6.2f} degrees")
print(f"mean |hue change|, V-only     : {np.abs(Hv-Hh).mean():6.2f} degrees")

---
## Think about it

1. The hue numbers above quantify what your eye already saw. Explain *mechanically* why
   per-channel equalisation moves hue, in terms of the three CDFs being different.
2. `C = 1 - R` converts RGB to CMY. So why do printers need a separate **K** (black) ink at all,
   if C+M+Y should already give black?
3. Hue is an **angle**. What goes wrong if you average the hues of two pixels at 350° and 10°?
   What is the correct way to average angles?
4. The V channel of HSV is `max(R,G,B)`; the Y of YCbCr is a weighted sum; the I of HSI is the
   plain mean. Compute all three for pure blue `(0,0,255)`. Which best matches how bright pure
   blue actually looks to you, and what does that say about HSV's V?
5. You want to detect a red ball under changing outdoor light. Which model, which channels,
   and which thresholds? Explain why an RGB box would fail.

In [ ]:
# Scratch space.

---
## Stretch

1. **HSI back to RGB.** Implement the inverse (the sector-based formulas in Gonzalez §6.2.4) and
   verify round-trip error is small. Then rotate every hue by 90° and look at the result.
2. **A skin detector.** Threshold in YCbCr (`77 < Cb < 127`, `133 < Cr < 173` is the classic box)
   and compare against your best RGB attempt on `messi5.jpg`. Report false positives on the grass.
3. **Colour distance.** Compute the distance between two similar colours in RGB and in Lab.
   Then find a pair where RGB says "far" and Lab says "near", and one the other way around.
4. **Vector median filter.** Median-filter a colour image per-channel and then again as vectors
   (the pixel whose summed distance to its neighbours is smallest). Add colour salt noise first.
   Per-channel invents colours that were never in the image; the vector version cannot.
5. **Bit-depth and palettes.** Quantise `fruits.jpg` to a 256-colour palette with k-means on the
   pixel colours, then display it as an indexed image plus a LUT. That is how GIF works.

In [ ]:
# Stretch scratch space.

---
## OpenCV API card - colour models and colormaps

```python
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
```
For `uint8`: **H in 0..179** (degrees ÷ 2), S and V in 0..255. For `float32`: H in 0..360,
S and V in 0..1. Getting this wrong is the most common HSV bug.

```python
mask = cv2.inRange(src, lowerb, upperb)     # -> uint8 0/255
```
The standard colour-mask idiom. Red **wraps** around H=0, so red needs *two* `inRange`
calls OR'd together.

```python
dst = cv2.applyColorMap(src, colormap)      # returns BGR
```
`COLORMAP_JET`, `VIRIDIS`, `TURBO`, `HOT`, `BONE`, `INFERNO`, `PLASMA`, `MAGMA`, `PARULA`.
A custom map is a `(256, 1, 3)` uint8 LUT in **BGR** order passed in place of the constant.

```python
b, g, r = cv2.split(src)
dst = cv2.merge([b, g, r])
```
Model cheat-sheet: **HSI** I = mean · **HSV** V = max · **HSL** L = (max+min)/2 ·
**YCrCb** Y = luma weights · **Lab** L = perceptual lightness.